# Quick Test — See Real Satellite Images + Train One Experiment
## Small sample from Rwanda's Nyungwe zone — runs in 10 minutes
This notebook downloads a small patch, shows the real satellite images,
and trains one Random Forest experiment so you can see results immediately.

## STEP 0 — Install and authenticate
Run this cell first. It will open a browser asking you to sign in with Google.

In [ ]:
# Install geemap if not already installed
import subprocess
subprocess.run(['pip', 'install', 'geemap', 'earthengine-api', '-q'])

import ee
import geemap

# This opens a browser — sign in with your Google account
# Only needed once on your computer
try:
    ee.Initialize()
    print('Already authenticated — good')
except:
    ee.Authenticate()
    ee.Initialize()
    print('Authentication done')

: 

## STEP 1 — Define Small Study Area
We use a 10km x 10km patch inside Nyungwe buffer zone.
Small enough to download fast, large enough to show real deforestation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

# Small 10km x 10km patch in Nyungwe buffer zone
# This area has documented deforestation events 2020-2024
study_patch = ee.Geometry.Rectangle([
    29.00, -2.75,   # southwest
    29.10, -2.65    # northeast
])

print('Study patch defined: ~10km x 10km in Nyungwe buffer zone')
print('Coordinates: 29.00-29.10 E,  -2.75 to -2.65 N')
print('This is inside Rwanda western province near Nyamasheke')

## STEP 2 — Load Sentinel-2 Images (Optical)
See the real satellite colour photograph of this Rwanda patch

In [ ]:
def mask_clouds(image):
    qa = image.select('QA60')
    cloud_mask = qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0))
    return image.updateMask(cloud_mask).divide(10000)

# Dry season 2021 (less cloud cover — clearer image)
s2_2021 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(study_patch)
           .filterDate('2021-06-01','2021-09-30')  # dry season
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
           .map(mask_clouds)
           .median()
           .clip(study_patch))

# Recent 2024 image
s2_2024 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(study_patch)
           .filterDate('2024-06-01','2024-09-30')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
           .map(mask_clouds)
           .median()
           .clip(study_patch))

# NDVI
ndvi_2021 = s2_2021.normalizedDifference(['B8','B4']).rename('NDVI_2021')
ndvi_2024 = s2_2024.normalizedDifference(['B8','B4']).rename('NDVI_2024')
ndvi_change = ndvi_2024.subtract(ndvi_2021).rename('NDVI_change')

print('Sentinel-2 images loaded for 2021 and 2024')
print('Computing NDVI...')

## STEP 3 — Load Sentinel-1 Radar
Radar works through clouds — this is what Rwanda looks like in radar

In [ ]:
s1_2021 = (ee.ImageCollection('COPERNICUS/S1_GRD')
           .filterBounds(study_patch)
           .filterDate('2021-01-01','2021-12-31')
           .filter(ee.Filter.eq('instrumentMode','IW'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
           .select(['VH','VV'])
           .median()
           .clip(study_patch))

s1_2024 = (ee.ImageCollection('COPERNICUS/S1_GRD')
           .filterBounds(study_patch)
           .filterDate('2024-01-01','2024-12-31')
           .filter(ee.Filter.eq('instrumentMode','IW'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
           .select(['VH','VV'])
           .median()
           .clip(study_patch))

print('Sentinel-1 radar images loaded for 2021 and 2024')
print('VH = vertical transmit, horizontal receive — sensitive to tree structure')

## STEP 4 — Show The Satellite Images
This downloads small thumbnail images so you can SEE what the data looks like

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def get_thumb(image, vis_params, region):
    """Download a small thumbnail of a GEE image"""
    url = image.getThumbURL({
        'region': region,
        'dimensions': 512,
        'format': 'png',
        **vis_params
    })
    response = requests.get(url)
    return Image.open(BytesIO(response.content))

region_geojson = study_patch.getInfo()

print('Downloading satellite image thumbnails...')
print('(This may take 30-60 seconds)')

# Sentinel-2 RGB (true colour)
s2_rgb_2021 = get_thumb(s2_2021, {'bands':['B4','B3','B2'],'min':0,'max':0.3}, region_geojson)
s2_rgb_2024 = get_thumb(s2_2024, {'bands':['B4','B3','B2'],'min':0,'max':0.3}, region_geojson)

# NDVI (green = forest, red = cleared)
ndvi_vis  = {'bands':['NDVI_2024'],'min':-0.2,'max':0.9,'palette':['#8B0000','#FFD700','#228B22']}
ndvi_2024_thumb = get_thumb(ndvi_2024, ndvi_vis, region_geojson)

# NDVI change (red = vegetation lost)
change_vis = {'bands':['NDVI_change'],'min':-0.5,'max':0.1,'palette':['#FF0000','#FF8C00','#FFFFFF']}
change_thumb = get_thumb(ndvi_change, change_vis, region_geojson)

# Sentinel-1 radar VH
s1_vh_2024 = get_thumb(s1_2024.select('VH'), {'min':-25,'max':-5,'palette':['#000000','#FFFFFF']}, region_geojson)

print('All images downloaded')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Real Satellite Images — Nyungwe Buffer Zone, Rwanda', 
             fontsize=14, fontweight='bold', y=0.98)

images_to_show = [
    (s2_rgb_2021,    'Sentinel-2 RGB\n2021 (True Colour)',    None),
    (s2_rgb_2024,    'Sentinel-2 RGB\n2024 (True Colour)',    None),
    (s1_vh_2024,     'Sentinel-1 Radar VH\n2024 (through clouds)', None),
    (ndvi_2024_thumb,'NDVI 2024\nDark green = forest, Red = cleared', None),
    (change_thumb,   'NDVI Change 2021→2024\nRed = forest lost (deforestation)', None),
]

for idx, (img, title, cmap) in enumerate(images_to_show):
    row, col = idx // 3, idx % 3
    axes[row][col].imshow(img)
    axes[row][col].set_title(title, fontsize=11, pad=8)
    axes[row][col].axis('off')

# Labels for the images
axes[0][0].set_xlabel('Green = forest   Brown = cleared land', fontsize=9)

# Hide unused subplot
axes[1][2].axis('off')
axes[1][2].text(0.5, 0.5,
    'What you are looking at:\n\n'
    'S2 RGB = colour photo from space\n'
    'S1 Radar = brightness = tree density\n'
    'NDVI = vegetation health index\n'
    'NDVI Change = where forest was lost\n\n'
    'Red patches in NDVI Change =\n'
    'deforestation events 2021-2024',
    ha='center', va='center', fontsize=10,
    transform=axes[1][2].transAxes,
    bbox=dict(boxstyle='round', facecolor='#F5F5F5', alpha=0.8))

plt.tight_layout()
plt.savefig('data/satellite_images_rwanda.png', dpi=150, bbox_inches='tight')
plt.show()
print('Image saved to data/satellite_images_rwanda.png')

## STEP 5 — Download Training Pixels From This Small Patch
Extract pixel values to train one experiment

In [ ]:
# Hansen labels for this small patch
hansen = ee.Image('UMD/hansen/global_forest_change_2023_v1_11').clip(study_patch)
srtm   = ee.Image('USGS/SRTMGL1_003').clip(study_patch)

# Loss label: 1 = lost forest 2020-2024
label = hansen.select('lossyear').gte(20).rename('label')
stable = hansen.select('treecover2000').gte(30).And(hansen.select('loss').eq(0))

# Feature stack
features = ee.Image.cat([
    ndvi_2021.rename('NDVI_2021'),
    ndvi_2024.rename('NDVI_2024'),
    ndvi_change.rename('NDVI_change'),
    s2_2021.select('B11').rename('SWIR_2021'),
    s1_2021.select('VH').rename('VH_2021'),
    s1_2021.select('VV').rename('VV_2021'),
    s1_2024.select('VH').rename('VH_2024'),
    s1_2024.select('VV').rename('VV_2024'),
    ee.Terrain.slope(srtm).rename('slope'),
    srtm.select('elevation'),
    label
]).float()

# Sample 1000 deforested + 1000 stable pixels
print('Sampling pixels from Rwanda satellite data...')
print('This may take 2-3 minutes...')

defor_pts = (features.updateMask(label)
             .sample(region=study_patch, scale=30, numPixels=1000, seed=42, geometries=False))

stable_pts = (features.updateMask(stable)
              .updateMask(label.eq(0))
              .sample(region=study_patch, scale=30, numPixels=1000, seed=42, geometries=False)
              .map(lambda f: f.set('label',0)))

combined = defor_pts.merge(stable_pts)

# Convert to pandas
data = combined.getInfo()
rows = [f['properties'] for f in data['features']]
df_mini = pd.DataFrame(rows).dropna()

print(f'Pixels downloaded: {len(df_mini)}')
print(f'Deforested: {(df_mini.label==1).sum()}')
print(f'Stable:     {(df_mini.label==0).sum()}')

df_mini.to_csv('data/mini_training_sample.csv', index=False)
print('Saved to data/mini_training_sample.csv')

## STEP 6 — Train One Quick Experiment (D: All Combined)
This is the real test — does it work on Rwanda data?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import time, os

feature_cols = [c for c in df_mini.columns if c != 'label']
X = df_mini[feature_cols].values
y = df_mini['label'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Training on {len(X_train)} pixels, testing on {len(X_test)} pixels')
print('Training Random Forest...')

start = time.time()
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print(f'Training done in {time.time()-start:.1f} seconds')

y_pred = model.predict(X_test)
f1 = f1_score(y_test, y_pred)

print()
print('=== MINI EXPERIMENT RESULT ===')
print(classification_report(y_test, y_pred,
      target_names=['Stable Forest','Deforested']))
print(f'F1-Score: {f1:.3f}')
print(f'Baseline to beat: CuSum-NRT = 0.71 (Ygorra et al. 2024)')

if f1 > 0.71:
    print(f'BEATS THE GLOBAL BASELINE — local calibration works for Rwanda')
elif f1 > 0.5:
    print(f'ABOVE RANDOM — small sample limits accuracy, full dataset will improve this')
else:
    print(f'LOW — likely too few deforestation pixels in this small patch')
    print(f'Try the full export in 01_GEE_Export.js for better results')

## STEP 7 — Feature Importance
Which satellite band matters most for THIS Rwanda patch?

In [ ]:
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(feature_cols)*0.4)))

colors = []
for f in importances['feature']:
    if 'VH' in f or 'VV' in f:
        colors.append('#1A5276')   # blue = radar
    elif f in ['slope','elevation']:
        colors.append('#7D3C98')   # purple = terrain
    else:
        colors.append('#1E6B3C')   # green = optical

ax.barh(importances['feature'], importances['importance'], color=colors)
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance — Rwanda Nyungwe Mini Sample\nWhich satellite band matters most?')

from matplotlib.patches import Patch
legend = [Patch(color='#1E6B3C', label='Sentinel-2 optical'),
          Patch(color='#1A5276', label='Sentinel-1 radar'),
          Patch(color='#7D3C98', label='SRTM terrain')]
ax.legend(handles=legend)

plt.tight_layout()
plt.savefig('data/mini_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

top = importances.tail(3)
print('Top 3 features for Rwanda Nyungwe:')
for _, r in top.iterrows():
    source = 'Radar' if ('VH' in r.feature or 'VV' in r.feature) \
             else 'Terrain' if r.feature in ['slope','elevation'] \
             else 'Optical'
    print(f'  {r.feature} ({source}): {r.importance:.4f}')

## STEP 8 — Interactive Map Showing What The Model Detects

In [ ]:
# Show interactive map with satellite layers
m = geemap.Map(center=[-2.70, 29.05], zoom=12)

# Add satellite imagery layers
m.addLayer(
    s2_2021,
    {'bands':['B4','B3','B2'], 'min':0, 'max':0.3},
    'Sentinel-2 RGB 2021'
)
m.addLayer(
    s2_2024,
    {'bands':['B4','B3','B2'], 'min':0, 'max':0.3},
    'Sentinel-2 RGB 2024'
)
m.addLayer(
    ndvi_change,
    {'min':-0.5, 'max':0.1,
     'palette':['red','orange','white']},
    'NDVI Change (Red = deforestation)'
)
m.addLayer(
    s1_2024.select('VH'),
    {'min':-25, 'max':-5,
     'palette':['black','white']},
    'Sentinel-1 Radar VH 2024'
)

# Add study area boundary
m.addLayer(
    ee.FeatureCollection([ee.Feature(study_patch)]),
    {'color':'yellow'},
    'Study Area'
)

m.add_layer_control()

# Save as HTML
m.save('data/interactive_satellite_map.html')
print('Interactive map saved to: data/interactive_satellite_map.html')
print('Open in your browser — toggle layers to see S2, Radar, NDVI change')

m

## Summary

In [ ]:
print('=== WHAT YOU JUST DID ===')
print()
print('1. Downloaded REAL Sentinel-2 optical images of Rwanda Nyungwe')
print('2. Downloaded REAL Sentinel-1 radar images of the same area')
print('3. Computed NDVI from actual satellite pixel values')
print('4. Downloaded real deforestation labels from Hansen GFC dataset')
print('5. Trained a Random Forest on real Rwanda satellite pixels')
print(f'6. Got F1-score: {f1:.3f}')
print()
print('Files saved:')
print('  data/satellite_images_rwanda.png  — visual of S2, S1, NDVI')
print('  data/mini_training_sample.csv     — real Rwanda pixel values')
print('  data/mini_feature_importance.png  — which band matters most')
print('  data/interactive_satellite_map.html — click to see Rwanda live')
print()
print('Next step: run 01_GEE_Export.js to get the FULL dataset')
print('Full dataset (10,000 pixels) will give much better accuracy.')